# Egypt Tech Jobs Pipeline — Simplified Demo

This notebook demonstrates the core data pipeline concept using 3 sources:
- **Workable** (company career pages)
- **Greenhouse** (company job boards)
- **Lever** (company career pages)

The current public implementation intentionally focuses on these three sources.

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import re
import time

## Configuration

In [ ]:
# Only fetch jobs posted in the last 24 hours
HOURS_BACK = 24
CUTOFF = datetime.utcnow() - timedelta(hours=HOURS_BACK)

# Company slugs/tokens to fetch from
WORKABLE_SLUGS = ['crossover', 'breadfast', 'paymob', 'valeo', 'swvl', 'fawry']
GREENHOUSE_TOKENS = ['andela', 'gitlab', 'twilio', 'mongodb', 'cloudflare', 'elastic']
LEVER_SLUGS = ['spotify', 'netflix']

print(f"Fetching jobs posted after: {CUTOFF.strftime('%Y-%m-%d %H:%M UTC')}")

## Source 1: Workable API

In [ ]:
def fetch_workable(slug):
    """Fetch jobs from a Workable company career page."""
    url = f"https://apply.workable.com/api/v1/widget/accounts/{slug}"
    jobs = []
    try:
        resp = requests.get(url, timeout=15)
        if resp.status_code != 200:
            return jobs
        data = resp.json()
        for job in data.get('jobs', []):
            published = job.get('published', '')
            if published:
                posted_dt = datetime.strptime(published[:19], '%Y-%m-%dT%H:%M:%S')
                if posted_dt < CUTOFF:
                    continue
            jobs.append({
                'title': job.get('title', ''),
                'company': slug.replace('-', ' ').title(),
                'location': job.get('location', ''),
                'work_type': 'Remote' if job.get('telecommuting', False) else 'On-site',
                'apply_url': f"https://apply.workable.com/{slug}/j/{job.get('shortcode', '')}/",
                'date_posted': published[:10] if published else '',
                'source': 'Workable'
            })
    except Exception as e:
        print(f"  Error fetching {slug}: {e}")
    return jobs

print("Fetching from Workable...")
workable_jobs = []
for slug in WORKABLE_SLUGS:
    result = fetch_workable(slug)
    workable_jobs.extend(result)
    print(f"  {slug}: {len(result)} jobs")
    time.sleep(1)

print(f"\nTotal Workable jobs (last 24h): {len(workable_jobs)}")

## Source 2: Greenhouse API

In [ ]:
def fetch_greenhouse(board_token):
    """Fetch jobs from a Greenhouse job board."""
    url = f"https://boards-api.greenhouse.io/v1/boards/{board_token}/jobs"
    jobs = []
    try:
        resp = requests.get(url, params={'content': 'true'}, timeout=15)
        if resp.status_code != 200:
            return jobs
        data = resp.json()
        for job in data.get('jobs', []):
            updated = job.get('updated_at', '')
            if updated:
                posted_dt = datetime.strptime(updated[:19], '%Y-%m-%dT%H:%M:%S')
                if posted_dt < CUTOFF:
                    continue
            location_name = ''
            if job.get('location'):
                location_name = job['location'].get('name', '')
            jobs.append({
                'title': job.get('title', ''),
                'company': board_token.replace('-', ' ').title(),
                'location': location_name,
                'work_type': 'Remote' if 'remote' in location_name.lower() else 'On-site',
                'apply_url': job.get('absolute_url', ''),
                'date_posted': updated[:10] if updated else '',
                'source': 'Greenhouse'
            })
    except Exception as e:
        print(f"  Error fetching {board_token}: {e}")
    return jobs

print("Fetching from Greenhouse...")
greenhouse_jobs = []
for token in GREENHOUSE_TOKENS:
    result = fetch_greenhouse(token)
    greenhouse_jobs.extend(result)
    print(f"  {token}: {len(result)} jobs")
    time.sleep(1)

print(f"\nTotal Greenhouse jobs (last 24h): {len(greenhouse_jobs)}")

## Source 3: Lever API

In [ ]:
def fetch_lever(company_slug):
    """Fetch jobs from a Lever career page."""
    url = f"https://api.lever.co/v0/postings/{company_slug}"
    jobs = []
    cutoff_ms = int(CUTOFF.timestamp() * 1000)
    try:
        resp = requests.get(url, params={'mode': 'json'}, timeout=15)
        if resp.status_code != 200:
            return jobs
        data = resp.json()
        for job in data:
            created_at = job.get('createdAt', 0)
            if created_at < cutoff_ms:
                continue
            location = ''
            categories = job.get('categories', {})
            if categories:
                location = categories.get('location', '')
            commitment = categories.get('commitment', '') if categories else ''
            jobs.append({
                'title': job.get('text', ''),
                'company': company_slug.replace('-', ' ').title(),
                'location': location,
                'work_type': 'Remote' if 'remote' in location.lower() else commitment if commitment else 'On-site',
                'apply_url': job.get('hostedUrl', ''),
                'date_posted': datetime.fromtimestamp(created_at / 1000).strftime('%Y-%m-%d') if created_at else '',
                'source': 'Lever'
            })
    except Exception as e:
        print(f"  Error fetching {company_slug}: {e}")
    return jobs

print("Fetching from Lever...")
lever_jobs = []
for slug in LEVER_SLUGS:
    result = fetch_lever(slug)
    lever_jobs.extend(result)
    print(f"  {slug}: {len(result)} jobs")
    time.sleep(1)

print(f"\nTotal Lever jobs (last 24h): {len(lever_jobs)}")

## Combine All Sources

In [ ]:
all_jobs = workable_jobs + greenhouse_jobs + lever_jobs
df = pd.DataFrame(all_jobs)

print(f"Total raw jobs fetched: {len(df)}")
print(f"\nBy source:")
if not df.empty:
    print(df['source'].value_counts().to_string())
else:
    print("  No jobs found in the last 24 hours (try running during business hours)")

## Data Quality Pipeline

In [ ]:
def clean_title(title):
    """Remove salary patterns and trailing noise from job titles."""
    title = re.sub(r'\$[\d,]+[kK]?\s*[-–]\s*\$[\d,]+[kK]?\s*(per\s+\w+)?', '', title)
    title = re.sub(r'\|.*$', '', title)
    title = re.sub(r'\s{2,}', ' ', title).strip()
    return title

def detect_level(title):
    """Detect seniority level from job title."""
    title_lower = title.lower()
    if any(kw in title_lower for kw in ['senior', 'sr.', 'sr ', 'lead', 'principal', 'staff']):
        return 'Senior'
    elif any(kw in title_lower for kw in ['junior', 'jr.', 'jr ', 'entry', 'associate', 'intern']):
        return 'Junior'
    elif any(kw in title_lower for kw in ['mid', 'intermediate', ' ii', ' iii']):
        return 'Mid'
    return 'Not specified'

SKILL_PATTERNS = [
    'python', 'java', 'javascript', 'typescript', 'react', 'angular', 'vue',
    'node\\.?js', 'django', 'flask', 'spring', 'docker', 'kubernetes', 'aws',
    'azure', 'gcp', 'sql', 'nosql', 'mongodb', 'postgresql', 'redis',
    'git', 'ci/cd', 'agile', 'scrum', 'machine learning', 'deep learning',
    'tensorflow', 'pytorch', 'nlp', 'data science', 'devops', 'linux',
    'c\\+\\+', 'c#', 'go', 'rust', 'swift', 'kotlin', 'ruby',
    'graphql', 'rest api', 'microservices', 'terraform', 'ansible'
]

def extract_skills(title, description=''):
    """Extract tech skills from title and description."""
    text = f"{title} {description}".lower()
    found = []
    for pattern in SKILL_PATTERNS:
        if re.search(r'\b' + pattern + r'\b', text):
            found.append(pattern.replace('\\', ''))
    return ', '.join(sorted(set(found))) if found else ''

if not df.empty:
    df['title'] = df['title'].apply(clean_title)
    df['level'] = df['title'].apply(detect_level)
    df['skills'] = df['title'].apply(lambda t: extract_skills(t))
    
    print("Quality pipeline applied.")
    print(f"\nLevel distribution:")
    print(df['level'].value_counts().to_string())
else:
    print("No data to process.")

## Deduplication

In [ ]:
if not df.empty:
    before = len(df)
    
    # URL-based deduplication
    df = df.drop_duplicates(subset=['apply_url'], keep='first')
    
    # Title + Company deduplication
    df['dedup_key'] = df['title'].str.lower().str.strip() + '||' + df['company'].str.lower().str.strip()
    df = df.drop_duplicates(subset=['dedup_key'], keep='first')
    df = df.drop(columns=['dedup_key'])
    
    after = len(df)
    print(f"Deduplication: {before} → {after} ({before - after} duplicates removed)")
else:
    print("No data to deduplicate.")

## Final Output

In [ ]:
if not df.empty:
    df = df.reset_index(drop=True)
    print(f"\nFinal dataset: {len(df)} jobs")
    print(f"\nSample (first 10):")
    display(df[['title', 'company', 'level', 'work_type', 'source']].head(10))
else:
    print("No jobs found. This is expected if no new jobs were posted in the last 24 hours.")
    print("Try again later or add more public company identifiers in the configuration cell.")